In [196]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import copy
import shapely
from shapely.geometry import LineString, Point, Polygon, MultiPolygon
from shapely.ops import split

ox.__version__

'2.0.3'

In [2]:
# Walkshed Setup

network_type = 'walk'
# custom filter for building walk network
cf = """
     ["area"!~"yes"]
     ["highway"]
     ["highway"!~"motor|proposed|construction|abandoned|platform|raceway"]
     ["foot"!~"no"]
     ["service"!~"private"]
     ["access"!~"private"]
     """
trip_times = [5, 10, 15]  # in minutes
travel_speed = 4.5  # walking speed in km/hour

# define a bounding box in Blacktown Suburbs as (left, bottom, right, top)
bbox = 150.868721,-33.766445,150.974464,-33.692731

In [3]:
# create network from that bounding box
graph = ox.graph_from_bbox(bbox, custom_filter=cf, network_type=network_type)

# save graph as GPKG and GraphML
ox.io.save_graph_geopackage(graph, filepath="walksheds/blacktown-walk-network.gpkg")
ox.io.save_graphml(graph, filepath="walksheds/blacktown-walk-network.graphml")

In [4]:
# Load graph from GraphML file
G = ox.io.load_graphml(filepath="walksheds/blacktown-walk-network.graphml")

In [ ]:
# # explore nodes and edges together in a single map
# nodes, edges = ox.graph_to_gdfs(G)
# m = edges.explore(color="skyblue", tiles="cartodbdarkmatter")
# nodes.explore(m=m, color="pink", marker_kwds={"radius": 6})

## Network Setup for Walksheds

In [6]:
# project the graph to UTM
G = ox.project_graph(G)

In [ ]:
# # add an edge attribute for time in minutes required to traverse each edge
# meters_per_minute = travel_speed * 1000 / 60  # km per hour to m per minute
# for _, _, _, data in G.edges(data=True, keys=True):
#     data["time"] = data["length"] / meters_per_minute

# get one color for each isochrone
# iso_colors = ox.plot.get_colors(n=len(trip_times), cmap="plasma", start=0)

In [87]:
# ------ Input points for walksheds ------
# suburb_stops_df = pd.read_csv("data/suburb_stops.csv")

# # convert stops to a geodataframe
# gdf_stops = gpd.GeoDataFrame(suburb_stops_df, geometry=gpd.points_from_xy(suburb_stops_df.lon, suburb_stops_df.lat), crs="EPSG:4326")

# # drop lat and lon from properties
# gdf_stops = gdf_stops.drop(columns=['lat', 'lon'])

# # Export to GeoJSON (before reprojecting to graph CRS)
# gdf_stops.to_file("walksheds/bus_stops.geojson", driver="GeoJSON")


# Read in bus stops geojson
gdf_stops = gpd.read_file("walksheds/bus_stops.geojson")

# Reproject stops to match graph CRS
gdf_stops = gdf_stops.to_crs(G.graph['crs'])

In [181]:
nearest_points = []

for idx, row in gdf_stops.iterrows():
    # Get nearest edge (returns a tuple: (u, v, key))
    nearest_edge = ox.distance.nearest_edges(G, X=row.geometry.x, Y=row.geometry.y)

    # Get the geometry of the nearest edge
    edge_data = G.get_edge_data(*nearest_edge)

    edge_geom = edge_data.get('geometry')

    # If edge has no geometry (some edges are just straight lines between nodes)
    if edge_geom is None:
        u = G.nodes[nearest_edge[0]]
        v = G.nodes[nearest_edge[1]]
        edge_geom = LineString([(u['x'], u['y']), (v['x'], v['y'])])

    origin_geom = row.geometry  # Ensure it's a shapely Point
    nearest_pt_on_edge = edge_geom.interpolate(edge_geom.project(origin_geom))

    # if nearest point is within 1 meter of an existing node, skip
    # find nearest node
    nearest_node = ox.distance.nearest_nodes(G, X=origin_geom.x, Y=origin_geom.y)

    # check distance between two
    # Get the coordinates of the nearest node
    nearest_node_coords = (G.nodes[nearest_node]["x"], G.nodes[nearest_node]["y"])

    # Measure the distance between the point and the nearest node
    distance_to_nearest_node = nearest_pt_on_edge.distance(Point(nearest_node_coords))

    # if distance is less than 0.5 meters
    if(distance_to_nearest_node < 0.5):
        print(f"Distance to nearest node within 0.5 meters, skipping (stop id: {row['stop_id']}, {distance_to_nearest_node} meters)")
        
        nearest_points.append({
            'stop_id': row['stop_id'],
            'node_id': int(nearest_node),
            'geometry': Point(nearest_node_coords)
        })

        continue

    # Append the geometry (Point), node_id, and stop_id as a dictionary
    nearest_points.append({
        'stop_id': row['stop_id'],
        'node_id': -1,
        'geometry': nearest_pt_on_edge
    })
        

# Create a GeoDataFrame from the list of features
gdf_nearest_points = gpd.GeoDataFrame(nearest_points)

# Optionally, assign an appropriate CRS (coordinate reference system) if necessary
gdf_nearest_points.set_crs(G.graph['crs'], inplace=True)

# Reproject to EPSG:4326 (WGS84 lat/lon)
#gdf_nearest_points_projected = gdf_nearest_points.to_crs("EPSG:4326")
# Export to GeoJSON
#gdf_nearest_points_projected.to_file("walksheds/nearest_point_on_edge.geojson", driver="GeoJSON")

Distance to nearest node within 0.5 meters, skipping (stop id: 2153389, 0.0 meters)
Distance to nearest node within 0.5 meters, skipping (stop id: 2768147, 0.0 meters)


,stop_id,node_id,geometry
0,2763180,-1,POINT (306302.357 6265392.534)
1,276327,-1,POINT (306018.545 6265636.845)
2,276365,-1,POINT (306853.059 6265332.132)
3,276366,-1,POINT (306555.426 6265299.078)
4,276367,-1,POINT (306303.471 6265374.6)
...,...,...,...
90,2768151,-1,POINT (308336.845 6266707.107)
91,2768152,-1,POINT (307603.619 6265999.6)
92,2768153,-1,POINT (307594.866 6265992.363)
93,2768154,-1,POINT (307257.419 6265254.691)


In [ ]:
# ------!!!! As of now, this block is dependent on the previous block being run -----
# ------!!!! for node_id to be set correctly in the dictionary                  -----

# -- Copy graph for safe editing
G_modified = G.copy()

# Copy geodataframe to prevent from having to regenerate each time
gdf_nearest_points_copy = copy.deepcopy(gdf_nearest_points)

# -- Track new nodes
new_node_ids = []

t_u = 0
t_v = 0

for idx, row in gdf_nearest_points_copy.iterrows():
    # if node_id has already been assigned, skip
    if row['node_id'] != -1:
        continue
    
    snapped_point = row.geometry
    stop_id = row.stop_id

    print("stop_id: " + str(stop_id))

    # 1. Find nearest edge
    u, v, key = ox.distance.nearest_edges(G_modified, X=snapped_point.x, Y=snapped_point.y)

    edge_data = G_modified.get_edge_data(u, v, key)
    edge_geom = edge_data.get("geometry")

    if edge_geom is None:
        point_u = (G_modified.nodes[u]["x"], G_modified.nodes[u]["y"])
        point_v = (G_modified.nodes[v]["x"], G_modified.nodes[v]["y"])
        edge_geom = LineString([point_u, point_v])


    # use a small linestring to split
    splitter = LineString([
        (snapped_point.x - 0.001, snapped_point.y),
        (snapped_point.x + 0.001, snapped_point.y)
    ])

    split_lines = split(edge_geom, splitter)

    # If split fails, skip
    if len(split_lines.geoms) != 2:
        print(f"Warning: Could not split edge {u}-{v} cleanly at stop_id {stop_id}")
        continue

    # remove the original edge(s)
    G_modified.remove_edge(u, v, key)
    
    # check if reciprocal edge is in graph, if so remove as well
    if((v, u) in G_modified.edges()):
        G_modified.remove_edge(v, u, key)

    # create new node
    new_node_id = max(G_modified.nodes) + 1
    G_modified.add_node(new_node_id, x=snapped_point.x, y=snapped_point.y, street_count=2) # street count is 2 since we're bisecting the existing edge
    
    # set node_id for newly created node in dataframe
    gdf_nearest_points_copy.at[idx, 'node_id'] = new_node_id
    
    u_to_node = False
    node_to_v = False

    make_u_next = False
    make_v_next = False

    for i, segment in enumerate(split_lines.geoms):
        attrs = edge_data.copy()
        attrs["geometry"] = segment
        attrs["length"] = segment.length

        segment_first_coord_point = Point(segment.coords[0])
        segment_last_coord_point = Point(segment.coords[len(segment.coords) - 1])

        u_node_point = Point(G_modified.nodes[u]["x"], G_modified.nodes[u]["y"])
        v_node_point = Point(G_modified.nodes[v]["x"], G_modified.nodes[v]["y"])

        if(i == 0):
            if(segment_first_coord_point == u_node_point or segment_last_coord_point == u_node_point):
                # create edge from u to new node
                # create in both directions to make sure it's bi-directional
                print(f"\tCreated first edge from {u} to {new_node_id}")
                G_modified.add_edge(u, new_node_id, **attrs)
                G_modified.add_edge(new_node_id, u, **attrs)
                make_v_next = True

            elif(segment_first_coord_point == v_node_point or segment_last_coord_point == v_node_point):
                G_modified.add_edge(v, new_node_id, **attrs)
                G_modified.add_edge(new_node_id, v, **attrs)
                print(f"\tCreated first edge from {v} to {new_node_id}")
                make_u_next = True
            else:
                # if the above logic falls through, find the node (u or v) that the first
                # segment coord is closest to, then create the edge from that node to the new node first
                u_to_segment_distance = shapely.distance(segment_first_coord_point, u_node_point)
                v_to_segment_distance = shapely.distance(segment_first_coord_point, v_node_point)

                print("\tSkipped first")

                if(u_to_segment_distance < v_to_segment_distance):
                    print(f"\tCreated first edge from {u} to {new_node_id}")
                    G_modified.add_edge(u, new_node_id, **attrs)
                    G_modified.add_edge(new_node_id, u, **attrs)
                    make_v_next = True
                else:
                    G_modified.add_edge(v, new_node_id, **attrs)
                    G_modified.add_edge(new_node_id, v, **attrs)
                    print(f"\tCreated first edge from {v} to {new_node_id}")
                    make_u_next = True

            
        elif(i == 1):
            if(make_v_next):
                # create edge from new node to v
                G_modified.add_edge(new_node_id, v, **attrs)
                G_modified.add_edge(v, new_node_id, **attrs)
                print(f"\tCreated second edge from {new_node_id} to {v}")
            elif(make_u_next):
                G_modified.add_edge(new_node_id, u, **attrs)
                G_modified.add_edge(u, new_node_id, **attrs)
                print(f"\tCreated second edge from {new_node_id} to {u}")
            else:
                print("\tSkipped second -------")
                print(f"\t\tU: {u}, V: {v}")
                print(f"\t\tNew Node: {new_node_id}")
            

stop_id: 2763180
	Created first edge from 12118059751 to 12850116610
	Created second edge from 12850116610 to 12118059723
stop_id: 276327
	Created first edge from 1690204475 to 12850116611
	Created second edge from 12850116611 to 2293531307
stop_id: 276365
	Created first edge from 12099801116 to 12850116612
	Created second edge from 12850116612 to 12099801125
stop_id: 276366
	Created first edge from 12118059755 to 12850116613
	Created second edge from 12850116613 to 12099801154
stop_id: 276367
	Created first edge from 12118059736 to 12850116614
	Created second edge from 12850116614 to 12118059755
stop_id: 276369
	Created first edge from 12099801210 to 12850116615
	Created second edge from 12850116615 to 12301445703
stop_id: 276859
	Created first edge from 12334306079 to 12850116616
	Created second edge from 12850116616 to 451673695
stop_id: 2763136
	Created first edge from 11914550057 to 12850116617
	Created second edge from 12850116617 to 11914550100
stop_id: 2763140
	Created first ed

In [200]:
# Save modified graph

# save graph as GPKG and GraphML
ox.io.save_graph_geopackage(G_modified, filepath="walksheds/blacktown-walk-network-modified.gpkg")
ox.io.save_graphml(G_modified, filepath="walksheds/blacktown-walk-network-modified.graphml")

In [199]:
# Setup Graph for walkshed creation

# add time in minutes to traverse each edge in the graph
meters_per_minute = travel_speed * 1000 / 60  # km per hour to m per minute
for _, _, _, data in G_modified.edges(data=True, keys=True):
    data["time"] = data["length"] / meters_per_minute

In [ ]:
# helper function for creating walksheds
def make_iso_polys(graph, origin, trip_times, stop_id, edge_buff=25, node_buff=50, infill=False):
    isochrone_polys = []

    # normalize trip_times to always be a list
    if isinstance(trip_times, (int, float)):
        trip_times = [trip_times]
    
    for trip_time in trip_times:
        subgraph = nx.ego_graph(graph, origin, radius=trip_time, distance="time")

        node_points = [Point((data["x"], data["y"])) for node, data in subgraph.nodes(data=True)]
        nodes_gdf = gpd.GeoDataFrame({"id": list(subgraph.nodes)}, geometry=node_points)
        nodes_gdf = nodes_gdf.set_index("id")

        edge_lines = []
        for n_fr, n_to in subgraph.edges():
            f = nodes_gdf.loc[n_fr].geometry
            t = nodes_gdf.loc[n_to].geometry

            edge_data = graph.get_edge_data(n_fr, n_to)
            if edge_data:
                # Safely get the first edge’s geometry, or fallback to straight line
                first_edge = list(edge_data.values())[0]
                edge_geom = first_edge.get("geometry", LineString([f, t]))
            else:
                edge_geom = LineString([f, t])

            edge_lines.append(edge_geom)
            
            # edge_lookup = graph.get_edge_data(n_fr, n_to)[0].get("geometry", LineString([f, t]))
            # edge_lines.append(edge_lookup)

        n = nodes_gdf.buffer(node_buff).geometry
        e = gpd.GeoSeries(edge_lines).buffer(edge_buff).geometry
        all_gs = list(n) + list(e)
        new_iso = gpd.GeoSeries(all_gs).union_all()

        if infill:
            if isinstance(new_iso, (MultiPolygon, Polygon)):
                new_iso = Polygon(new_iso.exterior)

        isochrone_polys.append(
        {
            'stop_id': stop_id,
            'walk_time': trip_time,
            'geometry': new_iso
        })

        # convert to a pandas df
        iso_df = pd.DataFrame(isochrone_polys, columns=['stop_id', 'walk_time', 'geometry'])

    return iso_df

## Create 400 meter buffers

In [210]:
time_to_walk_400m = 5.33333

# Process all stops
features = []
new_nodes = []

for idx, row in gdf_nearest_points_copy.iterrows():

    center_node = row['node_id']

    iso_polys = make_iso_polys(G_modified, center_node, time_to_walk_400m, row.stop_id, edge_buff=25, node_buff=0, infill=True)

    features.append(iso_polys)


print(features)

#new_nodes = pd.DataFrame(new_nodes, columns=['stop_id', 'lon', 'lat'])

# Create GeoDataFrame
#new_nodes_df = gpd.GeoDataFrame(new_nodes, geometry=gpd.points_from_xy(new_nodes.lon, new_nodes.lat), crs=G.graph['crs'])

# drop lat and lon columns
#new_nodes_df = new_nodes_df.drop(columns=['lon', 'lat'])

# Reproject to EPSG:4326 (WGS84 lat/lon)
#new_nodes_df = new_nodes_df.to_crs("EPSG:4326")

# Export to GeoJSON
#new_nodes_df.to_file("walksheds/stop_nodes.geojson", driver="GeoJSON")


# use concat with ignore index to remove duplicate column names and index from 0 to n-1
final_df = pd.concat(features, ignore_index=True)

# Create GeoDataFrame
iso_gdf = gpd.GeoDataFrame(final_df, crs=G.graph['crs'])

# Reproject to EPSG:4326 (WGS84 lat/lon)
iso_gdf = iso_gdf.to_crs("EPSG:4326")

print("outputting GeoJSON to 'walkshed_400m_new_TEST.geojson'")

# Export to GeoJSON
iso_gdf.to_file("walksheds/walkshed_400m_new_TEST.geojson", driver="GeoJSON")


subgraph
MultiDiGraph with 65 nodes and 178 edges
subgraph
MultiDiGraph with 96 nodes and 240 edges
subgraph
MultiDiGraph with 65 nodes and 170 edges
subgraph
MultiDiGraph with 75 nodes and 200 edges
subgraph
MultiDiGraph with 62 nodes and 168 edges
subgraph
MultiDiGraph with 34 nodes and 82 edges
subgraph
MultiDiGraph with 83 nodes and 220 edges
subgraph
MultiDiGraph with 45 nodes and 118 edges
subgraph
MultiDiGraph with 71 nodes and 180 edges
subgraph
MultiDiGraph with 77 nodes and 204 edges
subgraph
MultiDiGraph with 70 nodes and 190 edges
subgraph
MultiDiGraph with 42 nodes and 106 edges
subgraph
MultiDiGraph with 77 nodes and 204 edges
subgraph
MultiDiGraph with 67 nodes and 182 edges
subgraph
MultiDiGraph with 74 nodes and 192 edges
subgraph
MultiDiGraph with 79 nodes and 198 edges
subgraph
MultiDiGraph with 44 nodes and 110 edges
subgraph
MultiDiGraph with 80 nodes and 200 edges
subgraph
MultiDiGraph with 62 nodes and 170 edges
subgraph
MultiDiGraph with 48 nodes and 124 edges
s

In [ ]:
# Import bus stop inventory csv
stop_inventory_df = pd.read_csv("data/suburb_bus_stop_inventory.csv")

# calculate number of amenities and create new field with this information
for indx, val in stop_inventory_df.iterrows():
    stop_inventory_df['num_amenities'] = stop_inventory_df.iloc[:, :].eq('Yes').sum(axis=1)

# join to suburb_stops dataframe to get lat/lon for each stop
stop_inventory_df = pd.merge(stop_inventory_df, suburb_stops_df[['stop_id', 'lat', 'lon']], on='stop_id', how='left')

# convert to geodataframe
gdf_stop_inventory = gpd.GeoDataFrame(stop_inventory_df, geometry=gpd.points_from_xy(stop_inventory_df.lon, stop_inventory_df.lat), crs="EPSG:4326")

# drop lat and lon from properties
gdf_stop_inventory = gdf_stop_inventory.drop(columns=['lat', 'lon'])

# export to GeoJSON
gdf_stop_inventory.to_file("walksheds/bus_stop_inventory.geojson", driver="GeoJSON")

'\n# join to suburb_stops dataframe to get lat/lon for each stop\nstop_inventory_df = pd.merge(stop_inventory_df, suburb_stops_df[[\'stop_id\', \'lat\', \'lon\']], on=\'stop_id\', how=\'left\')\n\n# convert to geodataframe\ngdf_stop_inventory = gpd.GeoDataFrame(stop_inventory_df, geometry=gpd.points_from_xy(stop_inventory_df.lon, stop_inventory_df.lat), crs="EPSG:4326")\n\n# drop lat and lon from properties\ngdf_stop_inventory = gdf_stop_inventory.drop(columns=[\'lat\', \'lon\'])\n\n# export to GeoJSON\ngdf_stop_inventory.to_file("walksheds/bus_stop_inventory.geojson", driver="GeoJSON")\n'

In [206]:
node_id = 12850116617

neighbors = list(G_modified.neighbors(node_id))

neighbors

[11914550057, 11914550100]

## SAVING FOR LATER IF NEEDED, ISOCHRONES

In [ ]:
print("creating walkshed polygons...")

# Process all stops
features = []

# for each stop, generate a node on the closest edge, then pass that network into the make isopoly function
for idx, row in gdf_stops.iterrows():
    center_node = ox.distance.nearest_nodes(G_modified, row.geometry.x, row.geometry.y)
    iso_polys = make_iso_polys(G_modified, center_node, trip_times, row.stop_id, row.suburb_name, edge_buff=25, node_buff=0, infill=True)

    features.append(iso_polys)
    

print("done.")

# use concat with ignore index to remove duplicate column names and index from 0 to n-1
final_df = pd.concat(features, ignore_index=True)

# Create GeoDataFrame
iso_gdf = gpd.GeoDataFrame(final_df, crs=G.graph['crs'])

# Reproject to EPSG:4326 (WGS84 lat/lon)
iso_gdf = iso_gdf.to_crs("EPSG:4326")

print("outputting GeoJSON to 'bus_stop_isochrones.geojson'")

# Export to GeoJSON
iso_gdf.to_file("walksheds/bus_stop_isochrones.geojson", driver="GeoJSON")